In [12]:
import requests
from minsearch import Index

DOCS_URL = "https://raw.githubusercontent.com/alexeygrigorev/llm-rag-workshop/main/notebooks/documents.json"
raw = requests.get(DOCS_URL, timeout=30).json()

def chunk_text(text, size=1500, overlap=300):
    if len(text) <= size:
        return [text]
    out, start = [], 0
    while start < len(text):
        out.append(text[start:start + size])
        start += size - overlap
    return out

chunks = []
for course in raw:
    cname = course["course"]
    for doc in course["documents"]:
        body = (
            f"Section: {doc.get('section', '')}\n"
            f"Question: {doc.get('question', '')}\n\n"
            f"{doc.get('text', '')}"
        )
        fname = f"{cname}/{doc.get('section', 'general')}".replace(" ", "_")
        for i, ch in enumerate(chunk_text(body)):
            chunks.append({"filename": f"{fname}#chunk{i}", "content": ch})

chunk_index = Index(text_fields=["content"], keyword_fields=[])
chunk_index.fit(chunks)
print(f"chunk_index ready: {len(chunks)} chunks indexed")

chunk_index ready: 998 chunks indexed


In [13]:
def search(query: str) -> str:
    """Searches the course database, returns the top 3 matching chunks as text."""
    results = chunk_index.search(query=query, num_results=3)
    formatted_results = []
    for doc in results:
        formatted_results.append(f"Source: {doc['filename']}\nContent:\n{doc['content']}")
    return "\n\n---\n\n".join(formatted_results)

In [14]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
client = Anthropic(api_key=os.getenv("CLAUDE_API_KEY"))

tools_schema = [
    {
        "name": "search",
        "description": (
            "Searches the course lesson database for information matching keywords. "
            "Use this tool multiple times with different keywords to gather complete context before answering."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "The search keyword or terms."}
            },
            "required": ["query"],
        },
    }
]

messages = [{"role": "user", "content": "How does the agentic loop work, and how is it different from plain RAG?"}]
system_instruction = (
    "You're a course teaching assistant. Answer the student's question using the search tool. "
    "Make multiple searches with different keywords before answering."
)

tool_call_count = 0
max_iterations = 10
print("Starting Agentic Loop with Claude...\n")

for i in range(max_iterations):
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=2000,
        system=system_instruction,
        tools=tools_schema,
        messages=messages,
    )
    messages.append({"role": "assistant", "content": response.content})

    tool_use_blocks = [block for block in response.content if block.type == "tool_use"]
    if not tool_use_blocks:
        print("\n--- Final Answer from Claude ---")
        for block in response.content:
            if block.type == "text":
                print(block.text)
        break

    tool_responses = []
    for tool_use in tool_use_blocks:
        if tool_use.name == "search":
            tool_call_count += 1
            query_param = tool_use.input.get("query")
            print(f"[{tool_call_count}] Claude triggers search: '{query_param}'")
            tool_result = search(query_param)
            tool_responses.append({
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": tool_result,
            })

    messages.append({"role": "user", "content": tool_responses})

print(f"\nTotal times the search tool was called: {tool_call_count}")

Starting Agentic Loop with Claude...

[1] Claude triggers search: 'agentic loop'
[2] Claude triggers search: 'plain RAG retrieval augmented generation'
[3] Claude triggers search: 'agentic AI agent loop LLM'
[4] Claude triggers search: 'RAG vs agent difference'
[5] Claude triggers search: 'agent tool use reasoning LLM steps'
[6] Claude triggers search: 'retrieval augmented generation pipeline search generate'
[7] Claude triggers search: 'agentic RAG LLM zoomcamp'
[8] Claude triggers search: 'function calling agent observe act plan'
[9] Claude triggers search: 'LLM zoomcamp agent module'
[10] Claude triggers search: 'RAG pipeline search knowledge base context generation'

--- Final Answer from Claude ---
Unfortunately, after multiple searches across different relevant keyword combinations, the course lesson database does not contain specific content on the **agentic loop** or **plain RAG** as topics. However, based on well-established course material and general knowledge in this domain